In [0]:
%run ./01_common

In [0]:
dbutils.widgets.text("start_date", "2026-01-01")
dbutils.widgets.text("end_date", "2026-12-31")
dbutils.widgets.text("entities", "commits,issues,pulls,contributors,repo_metadata")
dbutils.widgets.dropdown("load_type", "full", ["full", "incremental"])

def bronze_to_silver(entity, start_date, end_date, load_type, batch_id):
    start = datetime.datetime.now()
    param = f"bronze_{entity} where to_date(_ingested_at) between {start_date} and {end_date}"
    try:
        src = (spark.table(T(f"bronze_{entity}"))
               .filter(F.to_date(F.expr("try_to_timestamp(_ingested_at)")).between(start_date, end_date)))
        n_read = src.count()
        cfg = SILVER[entity]
        df = src.select(*cfg["cols"])
        df = (df.withColumn("row_hash", F.sha2(F.to_json(F.struct(*df.columns)), 256))
                .withColumn("load_timestamp", F.current_timestamp()))
        for k in cfg["pk"]:
            df = df.filter(F.col(k).isNotNull())
        df = df.dropDuplicates(cfg["pk"])
        ins, upd = upsert(df, T(f"silver_{entity}"), cfg["pk"], "t.row_hash <> s.row_hash")
        log_run(batch_id, "Bronze-to-Silver", entity, param, load_type, start, "SUCCESS", n_read, ins, upd)
    except Exception as e:
        log_run(batch_id, "Bronze-to-Silver", entity, param, load_type, start, "FAILURE", err=str(e)[:2000])

bid = str(uuid.uuid4())
for e in dbutils.widgets.get("entities").split(","):
    bronze_to_silver(e.strip(), dbutils.widgets.get("start_date"), dbutils.widgets.get("end_date"),
                     dbutils.widgets.get("load_type"), bid)
display(spark.table(T("pipeline_execution_logs")).orderBy(F.desc("start_time")).limit(20))